# EnderLeaf script preparation

## Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# Needed to import from the enderscope library

import os

os.chdir("..")

## Imports

In [ ]:
from rich.pretty import pprint

import numpy as np
import cv2
import pandas as pd

import panel as pn

from libcamera import controls

from enderleaf.image import to_pil, canny, find_circles
from enderleaf.preview_panel import preview, extract_metadata, expand_file_path

## Initialize Preview

In [ ]:
preview().show()

In [ ]:
preview().on_center_on_qr_code()

In [ ]:
controls = {
    "AeEnable": False,
    "ExposureTime": 7000,  # 15ms exposure
    "AnalogueGain": 1,  # Manual gain (~ISO 400-800 depending on sensor)
    "AwbEnable": False,
    "ColourGains": (2.4, 1.0),  # Red & Blue gains for warm/cool WB
}

controls = {
    "AeEnable": True,
    "AwbEnable": True,
}



preview().camera.set_controls(controls)
# preview().camera.set_controls({"ExposureValue":1.5})
# preview().camera.set_controls({"ColourTemperature":7590})

In [ ]:
pprint(dict(sorted(preview().camera.capture_metadata().items())))

In [ ]:
pd.DataFrame(extract_metadata(preview().camera.capture_metadata()))

In [ ]:
from datetime import datetime as dt

dt.now().strftime("%Y%m%d%H%M%S")

In [ ]:
preview().stop()

## Grab circles

In [ ]:
factor = 4
idx = 3

In [ ]:
image = images[idx]["image"]
image = cv2.resize(image, (image.shape[1] // factor, image.shape[0] // factor))
edges = canny(image=image, color_space="rgb", channel="blue")
to_pil(edges)

In [ ]:
accums, cx, cy, radii = find_circles(
    edges=edges,
    radii=np.arange(450 // factor, 550 // factor, 20 // factor),
    max_circles=3,
)
pprint(accums, cx, cy, radii)
for cy, cx, radius in zip(cy, cx, radii):
    print(radius * factor)
    image = cv2.circle(
        image, (cx, cy), radius, (255, 0, 255), 10
    )

to_pil(image)